In [1]:
import requests

api_url = "https://115.159.159.82/api/devices"
r = requests.get(
    api_url,
    params={
        "page":1,"page_size":50
    },
    timeout=10
)

print("status:", r.status_code)
print("url:", r.url)
print("content-type:", r.headers.get("Content-Type"))

status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=50
content-type: application/json


In [2]:
data = r.json()

print(type(data))
print(data.keys())


<class 'dict'>
dict_keys(['page', 'page_size', 'total', 'items'])


In [3]:
print(data)
print(data["items"][0])

{'page': 1, 'page_size': 50, 'total': 5000, 'items': [{'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}, {'device_id': 'D00000002', 'company_id': 'C000001', 'device_name': '循环泵-000001-002', 'device_type': 'pump', 'model': 'SP-PUM-500', 'zone': '智造一区', 'install_date': '2016-03-06', 'rated_power_kw': 73.8, 'sampling_interval_s': 60, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-08T05:44:00'}, {'device_id': 'D00000003', 'company_id': 'C000001', 'device_name': '空压机-000001-003', 'device_type': 'compressor', 'model': 'SP-COM-200', 'zone': '智造一区', 'install_date': '2016-12-18', 'rated_power_kw': 248.6, 'sampling_interval_s': 30, 'protocol': 'modbus', 'status': 'running', 'updated_at': '2026-08-29T04:01:00'}, {'device_id': 'D

In [4]:
total = data["total"]
page_size = data["page_size"]
total_pages = (total + page_size - 1) // page_size
print("total:", total)
print("page_size:", page_size)
print("total_pages:", total_pages)

total: 5000
page_size: 50
total_pages: 100


In [6]:
devices = []
for page in range(1, total_pages + 1):
    r = requests.get(
        api_url,
        params={
            "page": page,
            "page_size": page_size
        },
        timeout=10
    )
    page_data = r.json()
    devices.extend(page_data["items"])
print("device count:", len(devices))

device count: 5000


In [7]:
import requests
from bs4 import BeautifulSoup

url = "https://115.159.159.82/companies"
resp = requests.get(url)
soup = BeautifulSoup(resp.text, "html.parser")

cards = soup.select("article.company-card")
card = cards[0]

company_id = card["data-company-id"]
name = card.select_one("h2 a").get_text(strip=True)
status = card.select_one(".badge").get_text(strip=True)

dl = card.select_one("dl")
dds = dl.select("dd")
industry = dds[0].get_text(strip=True)
zone = dds[1].get_text(strip=True)
device_count = dds[3].get_text(strip=True)

print("company_id:", company_id)
print("name:", name)
print("status:", status)
print("industry:", industry)
print("zone:", zone)
print("device_count:", device_count)

company_id: C000001
name: 临港精工001有限公司
status: 运营中
industry: 汽车零部件
zone: 智造一区
device_count: 6 台


In [8]:
all_companies = []
for card in cards:
    company_id = card["data-company-id"]
    name = card.select_one("h2 a").get_text(strip=True)
    status = card.select_one(".badge").get_text(strip=True)
    dl = card.select_one("dl")
    dds = dl.select("dd")
    industry = dds[0].get_text(strip=True)
    zone = dds[1].get_text(strip=True)
    device_count = dds[3].get_text(strip=True)
    
    info = {
        "company_id": company_id,
        "name": name,
        "status": status,
        "industry": industry,
        "zone": zone,
        "device": device_count
    }
    all_companies.append(info)

print("当前页面采集企业数量：", len(all_companies))
all_companies


当前页面采集企业数量： 20


[{'company_id': 'C000001',
  'name': '临港精工001有限公司',
  'status': '运营中',
  'industry': '汽车零部件',
  'zone': '智造一区',
  'device': '6 台'},
 {'company_id': 'C000002',
  'name': '临港装备002有限公司',
  'status': '运营中',
  'industry': '精密电子',
  'zone': '智造二区',
  'device': '26 台'},
 {'company_id': 'C000003',
  'name': '临港动力003有限公司',
  'status': '运营中',
  'industry': '精密电子',
  'zone': '智造一区',
  'device': '6 台'},
 {'company_id': 'C000004',
  'name': '临港机电004有限公司',
  'status': '运营中',
  'industry': '精密电子',
  'zone': '能源中心',
  'device': '6 台'},
 {'company_id': 'C000005',
  'name': '临港能源005有限公司',
  'status': '运营中',
  'industry': '智能制造',
  'zone': '能源中心',
  'device': '18 台'},
 {'company_id': 'C000006',
  'name': '临港新材006有限公司',
  'status': '运营中',
  'industry': '新能源',
  'zone': '能源中心',
  'device': '7 台'},
 {'company_id': 'C000007',
  'name': '临港科技007有限公司',
  'status': '运营中',
  'industry': '新能源',
  'zone': '能源中心',
  'device': '9 台'},
 {'company_id': 'C000008',
  'name': '临港智造008有限公司',
  'status': '运营中',
  'industry

In [9]:
base_url = "https://115.159.159.82/companies"
all_companies = []
page = 1

while page <=3:
    print(f"正在采集第 {page} 页")
    resp = requests.get(base_url, params={"page":page})
    soup = BeautifulSoup(resp.text, "html.parser")
    cards = soup.select("article.company-card")
    
    for card in cards:
        company_id = card["data-company-id"]
        name = card.select_one("h2 a").get_text(strip=True)
        status = card.select_one(".badge").get_text(strip=True)
        dl = card.select_one("dl")
        dds = dl.select("dd")
        industry = dds[0].get_text(strip=True)
        zone = dds[1].get_text(strip=True)
        device_count = dds[3].get_text(strip=True)
        
        info = {
            "company_id": company_id,
            "name": name,
            "status": status,
            "industry": industry,
            "zone": zone,
            "device": device_count
        }
        all_companies.append(info)
    page +=1

print(f"3页采集完成，一共{len(all_companies)}家企业")
all_companies


正在采集第 1 页
正在采集第 2 页
正在采集第 3 页
3页采集完成，一共60家企业


[{'company_id': 'C000001',
  'name': '临港精工001有限公司',
  'status': '运营中',
  'industry': '汽车零部件',
  'zone': '智造一区',
  'device': '6 台'},
 {'company_id': 'C000002',
  'name': '临港装备002有限公司',
  'status': '运营中',
  'industry': '精密电子',
  'zone': '智造二区',
  'device': '26 台'},
 {'company_id': 'C000003',
  'name': '临港动力003有限公司',
  'status': '运营中',
  'industry': '精密电子',
  'zone': '智造一区',
  'device': '6 台'},
 {'company_id': 'C000004',
  'name': '临港机电004有限公司',
  'status': '运营中',
  'industry': '精密电子',
  'zone': '能源中心',
  'device': '6 台'},
 {'company_id': 'C000005',
  'name': '临港能源005有限公司',
  'status': '运营中',
  'industry': '智能制造',
  'zone': '能源中心',
  'device': '18 台'},
 {'company_id': 'C000006',
  'name': '临港新材006有限公司',
  'status': '运营中',
  'industry': '新能源',
  'zone': '能源中心',
  'device': '7 台'},
 {'company_id': 'C000007',
  'name': '临港科技007有限公司',
  'status': '运营中',
  'industry': '新能源',
  'zone': '能源中心',
  'device': '9 台'},
 {'company_id': 'C000008',
  'name': '临港智造008有限公司',
  'status': '运营中',
  'industry

In [19]:
import requests
all_devices = []
page = 1

while page <= 100:
    api_url = "https://115.159.159.82/api/devices"
    res = requests.get(api_url, params={"page":page})
    data = res.json()
    items = data["items"]   # 这里改成 items
    for item in items:
        dev_info = {
            "device_id": item["device_id"],
            "company_id": item["company_id"],
            "device_name": item["device_name"],
            "status": item["status"]
        }
        all_devices.append(dev_info)
    print(f"已完成第{page}页")
    page +=1
    
print(f"✅设备采集完毕，本次采集设备数量：{len(all_devices)}")





已完成第1页
已完成第2页
已完成第3页
已完成第4页
已完成第5页
已完成第6页
已完成第7页
已完成第8页
已完成第9页
已完成第10页
已完成第11页
已完成第12页
已完成第13页
已完成第14页
已完成第15页
已完成第16页
已完成第17页
已完成第18页
已完成第19页
已完成第20页
已完成第21页
已完成第22页
已完成第23页
已完成第24页
已完成第25页
已完成第26页
已完成第27页
已完成第28页
已完成第29页
已完成第30页
已完成第31页
已完成第32页
已完成第33页
已完成第34页
已完成第35页
已完成第36页
已完成第37页
已完成第38页
已完成第39页
已完成第40页
已完成第41页
已完成第42页
已完成第43页
已完成第44页
已完成第45页
已完成第46页
已完成第47页
已完成第48页
已完成第49页
已完成第50页
已完成第51页
已完成第52页
已完成第53页
已完成第54页
已完成第55页
已完成第56页
已完成第57页
已完成第58页
已完成第59页
已完成第60页
已完成第61页
已完成第62页
已完成第63页
已完成第64页
已完成第65页
已完成第66页
已完成第67页
已完成第68页
已完成第69页
已完成第70页
已完成第71页
已完成第72页
已完成第73页
已完成第74页
已完成第75页
已完成第76页
已完成第77页
已完成第78页
已完成第79页
已完成第80页
已完成第81页
已完成第82页
已完成第83页
已完成第84页
已完成第85页
已完成第86页
已完成第87页
已完成第88页
已完成第89页
已完成第90页
已完成第91页
已完成第92页
已完成第93页
已完成第94页
已完成第95页
已完成第96页
已完成第97页
已完成第98页
已完成第99页
已完成第100页
✅设备采集完毕，本次采集设备数量：5000


In [11]:
import re
company_total = len(all_companies)
unique_company = len(set(c["company_id"] for c in all_companies))
device_total = len(all_devices)
unique_device = len(set(d["device_id"] for d in all_devices))
mismatch = 0

for comp in all_companies:
    cid = comp["company_id"]
    real_device_count = len([d for d in all_devices if d.get("company_id") == cid])
    card_num = int(re.search(r"\d+", comp["device"]).group())
    if real_device_count != card_num:
        mismatch += 1

print(f"Company 总数: {company_total}")
print(f"唯一 company_id 数: {unique_company}")
print(f"Device 总数: {device_total}")
print(f"唯一 device_id 数: {unique_device}")
print(f"数量不匹配的企业数量: {mismatch}")


Company 总数: 60
唯一 company_id 数: 60
Device 总数: 0
唯一 device_id 数: 0
数量不匹配的企业数量: 60


In [13]:
import json
with open("companies.json","w",encoding="utf-8") as f:
    json.dump(all_companies,f,ensure_ascii=False,indent=2)
    
with open("devices.json","w",encoding="utf-8") as f:
    json.dump(all_devices,f,ensure_ascii=False,indent=2)

print("文件保存成功")


文件保存成功


In [20]:
page_size = 50
final_device_count = len(all_devices)
unique_device_count = len(set(d["device_id"] for d in all_devices))

res = requests.get("https://115.159.159.82/api/devices", params={"page":1})
ret = res.json()
total = ret["total"]
total_page = (total + page_size -1 ) // page_size

print(f"每页条数: {page_size}")
print(f"设备总数: {total}")
print(f"程序计算得到的总页数: {total_page}")
print(f"最终采集设备数: {final_device_count}")
print(f"唯一 device_id 数: {unique_device_count}")


每页条数: 50
设备总数: 5000
程序计算得到的总页数: 100
最终采集设备数: 5000
唯一 device_id 数: 5000


In [18]:
import requests
res = requests.get("https://115.159.159.82/api/devices", params={"page":1})
data = res.json()
print(data)


{'page': 1, 'page_size': 50, 'total': 5000, 'items': [{'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}, {'device_id': 'D00000002', 'company_id': 'C000001', 'device_name': '循环泵-000001-002', 'device_type': 'pump', 'model': 'SP-PUM-500', 'zone': '智造一区', 'install_date': '2016-03-06', 'rated_power_kw': 73.8, 'sampling_interval_s': 60, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-08T05:44:00'}, {'device_id': 'D00000003', 'company_id': 'C000001', 'device_name': '空压机-000001-003', 'device_type': 'compressor', 'model': 'SP-COM-200', 'zone': '智造一区', 'install_date': '2016-12-18', 'rated_power_kw': 248.6, 'sampling_interval_s': 30, 'protocol': 'modbus', 'status': 'running', 'updated_at': '2026-08-29T04:01:00'}, {'device_id': 'D

In [22]:
import re
company_total = len(all_companies)
unique_company = len(set(c["company_id"] for c in all_companies))
device_total = len(all_devices)
unique_device = len(set(d["device_id"] for d in all_devices))
mismatch = 0
for comp in all_companies:
    cid = comp["company_id"]
    real_device_count = len([d for d in all_devices if d.get("company_id") == cid])
    # 这里改成 comp["device"]
    card_num = int(re.search(r"\d+", comp["device"]).group())
    if real_device_count != card_num:
        mismatch += 1
print(f"Company 总数: {company_total}")
print(f"唯一 company_id 数: {unique_company}")
print(f"Device 总数: {device_total}")
print(f"唯一 device_id 数: {unique_device}")
print(f"Company-Device 不一致企业数: {mismatch}")



Company 总数: 60
唯一 company_id 数: 60
Device 总数: 5000
唯一 device_id 数: 5000
Company-Device 不一致企业数: 0
